<a href="https://colab.research.google.com/github/iras-mpark/MLA1020/blob/main/week6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import torch
from typing import Any

In [6]:
class Step:
  """Represents taking an `action`, incurring some `cost` and ending up in a new `state`."""
  def __init__(self, action: Any, cost: float, state: Any):
    self.action = action
    self.cost = cost
    self.state = state

In [7]:
class SearchProblem:
  """Formally and fully represents a search problem."""
  def start_state(self) -> Any:
    raise NotImplementedError
  def successors(self, state: Any) -> list[Step]:
    raise NotImplementedError
  def is_end(self, state: Any) -> bool:
    raise NotImplementedError

# UCS Example

### Diamond problem formulation

In [8]:
class DiamondSearchProblem(SearchProblem):
  def __init__(self):
    # state -> state -> cost
    self.graph = {
      "A": {"B": 1, "C": 100},
      "B": {"A": 1, "C": 1, "D": 100},
      "C": {"A": 100, "B": 1, "D": 1},
      "D": {"B": 100, "C": 1},
    }
  def start_state(self) -> str:
    return "A"

  def successors(self, state: str) -> list[Step]:
    return [
      Step(action=new_state, cost=cost, state=new_state) \
      for new_state, cost in self.graph[state].items()
    ]

  def is_end(self, state: str) -> bool:
    return state == "D"

# UCS Algorithm

In [9]:
def uniform_cost_search(problem: SearchProblem) -> tuple[Solution | None, int]:
  """
  Run Uniform Cost Search (UCS) on the specified search `problem`.
  Return the solution (sequence of steps) and the number of states explored.
  """
  # Frontier: states we've seen, still trying to figure out how the best way to get there
  # Priority represents the minimum cost to get there
  frontier = PriorityQueue()
  # For each state we've reached, backpointer tells us how we got there
  backpointers: dict[Any, Backpointer] = {}
  num_explored = 0
  # Add the start state
  start_state = problem.start_state()
  frontier.update(start_state, 0.0)
  while True:
    # Remove the state from the frontier with the lowest priority (theorem: priority = past_cost).
    state, past_cost = frontier.remove_min()
    if state is None and past_cost is None:
      return None, num_explored  # Found no solution
    num_explored += 1
    # Check if we've reached an end state; if so, extract solution.
    if problem.is_end(state):
      # Walk back the backpointers to get the actions
      steps = []
      while state != start_state:
        backpointer = backpointers[state]
        steps.insert(0, Step(backpointer.action, backpointer.cost, state))  # Prepend
        state = backpointer.prev_state  # Go back
      return Solution(steps=steps), num_explored
    # Expand from `state`, updating the frontier with each `new_state`
    for successor in problem.successors(state):
      if frontier.update(successor.state, past_cost + successor.cost):
          # We found better way to get to `successor.state` --> update backpointer!
          backpointers[successor.state] = Backpointer(prev_state=state, action=successor.action, cost=successor.cost)

NameError: name 'Solution' is not defined